# Phase 1 — Spark Distributed Processing

This notebook visualizes the outputs generated by the Spark scripts:

- `01_clean_logs.py`
- `02_market_basket.py`
- `03_user_affinity.py`

The heavy processing was already done using Spark scripts. This notebook reads the generated outputs for presentation.

In [16]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("Phase 1 Results Notebook")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

## Cleaned Dataset

The raw CSV was cleaned and converted into Parquet format.

The cleaning step extracted JSON fields from:

- `user_metadata`
- `product_metadata`

and produced fields such as:

- `category`
- `brand`
- `device`
- `user_tier`
- `user_location`

In [17]:
clean_path = r"C:\BigData_Project2\output\clean_logs_parquet"

clean_df = spark.read.parquet(clean_path)

clean_df.printSchema()

root
 |-- event_timestamp: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- referrer: string (nullable = true)
 |-- device: string (nullable = true)
 |-- user_tier: string (nullable = true)
 |-- user_location: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- stock: integer (nullable = true)



In [18]:
display(clean_df.limit(10).toPandas())

,event_timestamp,session_id,user_id,event_type,item_id,price,referrer,device,user_tier,user_location,category,brand,stock
0,2026-04-05 06:59:47,SESS_37dff982b5,User_2689,view,ITEM_2883,1295.83,direct,Mobile,Platinum,FR,Home,IKEA,97
1,2026-01-03 23:37:56,SESS_4d063f4f92,User_15360,view,ITEM_1317,856.58,instagram.com,Desktop,Platinum,US,Books,Macmillan,49
2,2026-04-11 06:24:42,SESS_6b95b40bce,User_22463,view,ITEM_3467,605.62,facebook.com,Desktop,Gold,EG,Home,Wayfair,75
3,2026-01-31 12:17:18,SESS_333a963524,User_6,purchase,ITEM_2521,844.78,google.com,Mobile,Silver,US,Toys,Hasbro,37
4,2026-04-14 18:15:56,SESS_a8e8c2c9ab,User_11262,view,ITEM_1107,1231.02,bing.com,Mobile,Platinum,EG,Home,IKEA,6
5,2026-02-14 08:08:53,SESS_ae8c283c6d,User_15609,view,ITEM_2740,707.77,None,Tablet,Silver,US,Books,Penguin,91
6,2026-02-18 23:25:28,SESS_56d14fd94f,User_3484,view,ITEM_2732,364.97,facebook.com,Tablet,Platinum,EG,Home,IKEA,46
7,2026-03-14 05:15:56,SESS_06bfffbf05,User_822,view,ITEM_3893,131.74,newsletter,Desktop,Silver,FR,Electronics,Dell,31
8,2026-01-23 11:33:17,SESS_fadacfa74a,User_20594,view,ITEM_117,963.21,instagram.com,Desktop,Silver,CA,Books,Penguin,1
9,2026-01-23 01:00:02,SESS_20736e386a,User_18351,view,ITEM_4407,NaN,facebook.com,Desktop,Bronze,US,Books,Penguin,0


In [19]:
event_counts = clean_df.groupBy("event_type").count().orderBy(col("count").desc())
display(event_counts.toPandas())

,event_type,count
0,view,7311095
1,cart,1950321
2,purchase,487423


In [20]:
category_counts = clean_df.groupBy("category").count().orderBy(col("count").desc())
display(category_counts.toPandas())

,category,count
0,Home,1808779
1,Books,1776812
2,Electronics,1761552
3,Toys,1739910
4,Clothing,1685780
5,None,976006


## Task 1.1 — Market Basket Analysis

Market Basket Analysis finds items that are purchased together in the same session.

MapReduce idea:

- Map: emit `(session_id, item_id)` for purchase events
- Reduce: group by session, generate item pairs, count each pair globally

In [21]:
item_pairs_path = r"C:\BigData_Project2\output\item_pairs"

item_pairs = spark.read.parquet(item_pairs_path)

top_pairs = item_pairs.orderBy(col("pair_count").desc()).limit(20)
display(top_pairs.toPandas())

,item_1,item_2,pair_count
0,ITEM_3262,ITEM_857,2
1,ITEM_386,ITEM_4168,2
2,ITEM_13,ITEM_279,2
3,ITEM_2468,ITEM_376,2
4,ITEM_301,ITEM_3891,2
5,ITEM_1446,ITEM_2643,2
6,ITEM_1783,ITEM_4791,2
7,ITEM_2355,ITEM_4236,2
8,ITEM_1060,ITEM_2214,2
9,ITEM_2378,ITEM_4868,2


## Task 1.2 — User Affinity Aggregation

User affinity profiles are created using weighted scores:

| Event Type | Weight |
|---|---|
| view | 1 |
| cart | 3 |
| purchase | 5 |

Each user receives a ranked list of their top product categories.

In [22]:
profiles_path = r"C:\BigData_Project2\output\user_profiles"

profiles = spark.read.json(profiles_path)

display(profiles.limit(20).toPandas())

,top_categories,user_id
0,"[(Home, 150), (Clothing, 124), (Books, 113)]",User_0
1,"[(Home, 137), (Electronics, 120), (Clothing, 1...",User_1
2,"[(Home, 114), (Electronics, 112), (Books, 110)]",User_10
3,"[(Books, 102), (Home, 99), (Toys, 97)]",User_100
4,"[(Home, 126), (Electronics, 117), (Books, 114)]",User_1000
5,"[(Books, 152), (Toys, 151), (Home, 148)]",User_10000
6,"[(Electronics, 127), (Books, 115), (Home, 111)]",User_10001
7,"[(Electronics, 139), (Books, 118), (Toys, 117)]",User_10002
8,"[(Books, 132), (Home, 127), (Toys, 127)]",User_10003
9,"[(Home, 121), (Electronics, 109), (Clothing, 1...",User_10004


In [23]:
spark.stop()